# Lunar Final Analysis

**Source script:** `lunar_final_analysis.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
import math
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Lunar-focused exploratory analysis.")
    parser.add_argument(
        "--input",
        default="/Users/omaralneser/Documents/TYP/outputs/imputed_dataset_enriched.csv",
        help="Path to enriched input CSV.",
    )
    parser.add_argument(
        "--outdir",
        default="/Users/omaralneser/Documents/TYP/outputs/eda_v3/lunar_final",
        help="Output directory.",
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--bin-size-deg",
        type=int,
        default=15,
        choices=[10, 15],
        help="Circular incidence bin size (10 -> 36 bins, 15 -> 24 bins).",
    )
    return parser.parse_args()


def slugify(text: str) -> str:
    s = re.sub(r"[^A-Za-z0-9]+", "_", str(text).strip())
    s = re.sub(r"_+", "_", s).strip("_")
    return s.lower()


def detect_column(df: pd.DataFrame, candidates: Sequence[str], required: bool = False) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        lc = cand.lower()
        if lc in lower_map:
            return lower_map[lc]
    if required:
        raise ValueError(f"Required column not found. Candidates: {candidates}")
    return None


def to_clean_str(x: object) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def map_sex(raw: object) -> str:
    s = to_clean_str(raw)
    if s in {"f", "female", "weiblich", "w"}:
        return "female"
    if s in {"m", "male", "männlich", "maennlich"}:
        return "male"
    if s in {"", "unknown", "unbekannt", "missing", "na", "nan", "?", "none"}:
        return "unknown"
    return "unknown"


def map_neuter(raw: object) -> str:
    s = to_clean_str(raw)
    if s in {"neutered", "spayed", "castrated", "kastriert", "sterilisiert"}:
        return "neutered"
    if s in {"intact", "entire", "unkastriert", "nicht kastriert"}:
        return "intact"
    if s in {"", "unknown", "unbekannt", "missing", "na", "nan", "?", "none"}:
        return "unknown"
    return "unknown"


def circular_diff_deg(a: np.ndarray, center: float) -> np.ndarray:
    return ((a - center + 180.0) % 360.0) - 180.0


def wilson_ci(k: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    if n <= 0:
        return np.nan, np.nan
    p = k / n
    denom = 1.0 + (z * z) / n
    center = (p + (z * z) / (2.0 * n)) / denom
    margin = (z / denom) * math.sqrt((p * (1.0 - p) / n) + (z * z) / (4.0 * n * n))
    return max(0.0, center - margin), min(1.0, center + margin)


def rr_from_counts(case_n: int, exp_n: int, base_case: int, base_n: int) -> Dict[str, float]:
    risk = case_n / exp_n if exp_n else np.nan
    baseline = base_case / base_n if base_n else np.nan
    rr = risk / baseline if baseline and baseline > 0 else np.nan

    r_lo, r_hi = wilson_ci(case_n, exp_n)
    b_lo, b_hi = wilson_ci(base_case, base_n)
    rr_lo = r_lo / b_hi if b_hi and b_hi > 0 else np.nan
    rr_hi = r_hi / b_lo if b_lo and b_lo > 0 else np.nan

    return {
        "exposure_n": int(exp_n),
        "case_n": int(case_n),
        "risk": float(risk) if pd.notna(risk) else np.nan,
        "baseline_risk": float(baseline) if pd.notna(baseline) else np.nan,
        "rr": float(rr) if pd.notna(rr) else np.nan,
        "rr_ci_low": float(rr_lo) if pd.notna(rr_lo) else np.nan,
        "rr_ci_high": float(rr_hi) if pd.notna(rr_hi) else np.nan,
        "ci_excludes_1": bool(pd.notna(rr_lo) and pd.notna(rr_hi) and (rr_hi < 1.0 or rr_lo > 1.0)),
    }


def create_moon_category_from_angle(angle_deg: pd.Series) -> pd.Series:
    angle = angle_deg % 360.0
    conds = [
        ((angle >= 315) | (angle < 45)),
        ((angle >= 45) & (angle < 135)),
        ((angle >= 135) & (angle < 225)),
        ((angle >= 225) & (angle < 315)),
    ]
    labels = ["new", "waxing", "full", "waning"]
    out = np.select(conds, labels, default="unknown")
    return pd.Series(out, index=angle_deg.index)


def normalize_moon_category(raw_cat: pd.Series, angle_deg: pd.Series) -> pd.Series:
    if raw_cat is None:
        return create_moon_category_from_angle(angle_deg)
    mapped = raw_cat.astype(str).str.strip().str.lower()
    out = pd.Series(index=raw_cat.index, dtype="object")
    out.loc[mapped.str.contains("new")] = "new"
    out.loc[mapped.str.contains("wax")] = "waxing"
    out.loc[mapped.str.contains("full")] = "full"
    out.loc[mapped.str.contains("wan")] = "waning"
    missing = out.isna()
    out.loc[missing] = create_moon_category_from_angle(angle_deg).loc[missing]
    return out.fillna("unknown")


def prepare_dataset(input_path: Path) -> Tuple[pd.DataFrame, str]:
    df = pd.read_csv(input_path)

    outcome_col = detect_column(df, ["group", "diagnosis_group", "diagnosis"], required=True)
    date_col = detect_column(df, ["presentation_date", "date of presentation"], required=True)
    sex_col = detect_column(df, ["sex", "gender", "geschlecht"])
    neuter_col = detect_column(df, ["neuter status", "neuter", "neutered", "reproductive"])
    angle_col = detect_column(df, ["moon_phase_angle_deg", "moon_phase_angle", "moon_angle"], required=True)
    frac_col = detect_column(df, ["moon_phase_fraction_illuminated", "moon_fraction_illuminated"])
    cat_col = detect_column(df, ["moon_phase_category", "moon_category"])
    full_col = detect_column(df, ["is_full_moon", "full_moon"])
    new_col = detect_column(df, ["is_new_moon", "new_moon"])

    df["_outcome"] = df[outcome_col].astype(str)
    df["_date"] = pd.to_datetime(df[date_col], errors="coerce")
    df["_moon_angle"] = pd.to_numeric(df[angle_col], errors="coerce") % 360.0
    if frac_col:
        df["_moon_fraction"] = pd.to_numeric(df[frac_col], errors="coerce")
    else:
        df["_moon_fraction"] = np.nan

    df["moon_phase_sin"] = np.sin(np.deg2rad(df["_moon_angle"]))
    df["moon_phase_cos"] = np.cos(np.deg2rad(df["_moon_angle"]))
    df["_moon_category"] = normalize_moon_category(df[cat_col] if cat_col else None, df["_moon_angle"])

    if full_col:
        df["_is_full_moon"] = pd.to_numeric(df[full_col], errors="coerce").fillna(0).astype(int).clip(0, 1)
    else:
        df["_is_full_moon"] = (np.abs(circular_diff_deg(df["_moon_angle"].to_numpy(), 180.0)) <= 15.0).astype(int)
    if new_col:
        df["_is_new_moon"] = pd.to_numeric(df[new_col], errors="coerce").fillna(0).astype(int).clip(0, 1)
    else:
        df["_is_new_moon"] = (np.abs(circular_diff_deg(df["_moon_angle"].to_numpy(), 0.0)) <= 15.0).astype(int)

    for w in (15, 30):
        df[f"_near_new_{w}"] = (np.abs(circular_diff_deg(df["_moon_angle"].to_numpy(), 0.0)) <= float(w)).astype(int)
        df[f"_near_full_{w}"] = (np.abs(circular_diff_deg(df["_moon_angle"].to_numpy(), 180.0)) <= float(w)).astype(int)

    if sex_col:
        df["sex_mapped"] = df[sex_col].map(map_sex)
    else:
        df["sex_mapped"] = "unknown"
    if neuter_col:
        df["neuter_mapped"] = df[neuter_col].map(map_neuter)
    else:
        df["neuter_mapped"] = "unknown"
    df["sex_neuter"] = df["sex_mapped"] + "_" + df["neuter_mapped"]

    df["_doy"] = df["_date"].dt.dayofyear
    df["_doy_sin"] = np.sin(2.0 * np.pi * df["_doy"] / 365.25)
    df["_doy_cos"] = np.cos(2.0 * np.pi * df["_doy"] / 365.25)
    return df, outcome_col


def build_sex_neuter_distribution(df: pd.DataFrame, diseases: Sequence[str], out_path: Path) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    total_n = len(df)
    for stratum, sdf in df.groupby("sex_neuter", dropna=False):
        row: Dict[str, object] = {
            "sex_neuter": stratum,
            "total_n": int(len(sdf)),
            "pct_total": float(len(sdf) / total_n if total_n else np.nan),
            "reported_stratified": bool(len(sdf) >= 20),
        }
        for d in diseases:
            c = int((sdf["_outcome"] == d).sum())
            row[f"case_n__{d}"] = c
            row[f"risk__{d}"] = c / len(sdf) if len(sdf) else np.nan
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(["total_n", "sex_neuter"], ascending=[False, True])
    out.to_csv(out_path, index=False)
    return out


def compute_category_rr(
    df: pd.DataFrame,
    disease: str,
    stratum: Optional[str] = None,
) -> pd.DataFrame:
    w = df if stratum is None else df[df["sex_neuter"] == stratum]
    y = (w["_outcome"] == disease).astype(int)
    base_case = int(y.sum())
    base_n = int(len(w))
    rows: List[Dict[str, object]] = []
    for cat, g in w.groupby("_moon_category", dropna=False):
        case_n = int((g["_outcome"] == disease).sum())
        exp_n = int(len(g))
        r = rr_from_counts(case_n, exp_n, base_case, base_n)
        rows.append(
            {
                "analysis_level": "stratified_category" if stratum else "overall_category",
                "disease": disease,
                "stratum": stratum if stratum else "overall",
                "metric": "moon_category",
                "level": str(cat),
                "window_deg": np.nan,
                **r,
            }
        )
    return pd.DataFrame(rows)


def compute_window_rr(df: pd.DataFrame, disease: str) -> pd.DataFrame:
    y = (df["_outcome"] == disease).astype(int)
    base_case = int(y.sum())
    base_n = int(len(df))
    rows: List[Dict[str, object]] = []
    for w in (15, 30):
        for label, col in [("near_new", f"_near_new_{w}"), ("near_full", f"_near_full_{w}")]:
            mask = df[col] == 1
            exp_n = int(mask.sum())
            case_n = int(((df["_outcome"] == disease) & mask).sum())
            r = rr_from_counts(case_n, exp_n, base_case, base_n)
            rows.append(
                {
                    "analysis_level": "overall_window",
                    "disease": disease,
                    "stratum": "overall",
                    "metric": label,
                    "level": "yes",
                    "window_deg": int(w),
                    **r,
                }
            )
    return pd.DataFrame(rows)


def circular_curve_table(df: pd.DataFrame, disease: str, bin_size: int) -> pd.DataFrame:
    edges = np.arange(0, 360 + bin_size, bin_size)
    bins = pd.cut(df["_moon_angle"], bins=edges, right=False, include_lowest=True)
    y = (df["_outcome"] == disease).astype(int)
    recs: List[Dict[str, float]] = []
    for i, interval in enumerate(bins.cat.categories):
        mask = bins == interval
        n = int(mask.sum())
        k = int(y[mask].sum())
        risk = k / n if n else np.nan
        lo, hi = wilson_ci(k, n)
        center = (edges[i] + edges[i + 1]) / 2.0
        recs.append(
            {
                "bin_center_deg": center,
                "bin_start_deg": float(edges[i]),
                "bin_end_deg": float(edges[i + 1]),
                "total_n": n,
                "case_n": k,
                "risk": risk,
                "risk_ci_low": lo,
                "risk_ci_high": hi,
            }
        )
    out = pd.DataFrame(recs).sort_values("bin_center_deg")
    arr = out["risk"].to_numpy()
    if len(arr) >= 3:
        arr_pad = np.r_[arr[-1], arr, arr[0]]
        smooth = np.convolve(arr_pad, np.ones(3) / 3.0, mode="valid")
        out["risk_smooth_ma3"] = smooth
    else:
        out["risk_smooth_ma3"] = out["risk"]
    return out


def save_circular_plot(curve: pd.DataFrame, disease: str, bin_size: int, out_png: Path) -> None:
    x = curve["bin_center_deg"].to_numpy()
    y = curve["risk"].to_numpy()
    lo = curve["risk_ci_low"].to_numpy()
    hi = curve["risk_ci_high"].to_numpy()
    ys = curve["risk_smooth_ma3"].to_numpy()

    plt.figure(figsize=(8.2, 4.4))
    plt.plot(x, y, marker="o", lw=1.1, color="#1f77b4", label="Per-bin risk")
    plt.fill_between(x, lo, hi, color="#1f77b4", alpha=0.18, label="Wilson 95% CI")
    plt.plot(x, ys, lw=2.0, color="#d62728", label="Moving average (window=3)")
    plt.title(f"Lunar Circular Incidence: {disease}")
    plt.xlabel(f"Moon phase angle (deg, bin size={bin_size})")
    plt.ylabel("Case risk (OvR)")
    plt.xlim(0, 360)
    plt.xticks(np.arange(0, 361, 45))
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def save_rr_barplot(df_rr: pd.DataFrame, title: str, x_col: str, out_png: Path) -> None:
    d = df_rr.copy()
    d = d.sort_values(x_col)
    x = np.arange(len(d))
    rr = d["rr"].to_numpy()
    lo = d["rr_ci_low"].to_numpy()
    hi = d["rr_ci_high"].to_numpy()
    yerr = np.vstack([np.maximum(rr - lo, 0), np.maximum(hi - rr, 0)])

    plt.figure(figsize=(8.2, 4.4))
    plt.bar(x, rr, color="#4c78a8", alpha=0.88)
    plt.errorbar(x, rr, yerr=yerr, fmt="none", color="black", capsize=3, lw=0.9)
    plt.axhline(1.0, color="#d62728", ls="--", lw=1.2)
    plt.xticks(x, d[x_col].astype(str), rotation=0)
    plt.ylabel("Risk ratio vs baseline")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def save_stratified_heatmap(
    df: pd.DataFrame,
    disease: str,
    strata_order: Sequence[str],
    out_png: Path,
) -> None:
    cols = ["new", "waxing", "full", "waning"]
    mat = np.full((len(strata_order), len(cols)), np.nan, dtype=float)
    ann = np.full((len(strata_order), len(cols)), "", dtype=object)

    for i, st in enumerate(strata_order):
        w = df[(df["disease"] == disease) & (df["stratum"] == st)]
        for j, col in enumerate(cols):
            row = w[w["level"] == col]
            if row.empty:
                continue
            r = row.iloc[0]
            mat[i, j] = r["rr"]
            ann[i, j] = f"{r['rr']:.2f}\n(n={int(r['exposure_n'])})" if pd.notna(r["rr"]) else ""

    if np.isfinite(mat).sum() == 0:
        return

    finite = mat[np.isfinite(mat)]
    vmin = min(0.5, float(np.nanmin(finite)))
    vmax = max(1.5, float(np.nanmax(finite)))

    plt.figure(figsize=(8.8, max(3.6, 0.45 * len(strata_order) + 1.8)))
    im = plt.imshow(mat, cmap="RdBu_r", aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(im, label="Risk ratio")
    plt.xticks(np.arange(len(cols)), cols)
    plt.yticks(np.arange(len(strata_order)), strata_order)
    plt.title(f"Stratified Lunar RR Heatmap: {disease}")
    for i in range(len(strata_order)):
        for j in range(len(cols)):
            if ann[i, j]:
                plt.text(j, i, ann[i, j], ha="center", va="center", fontsize=7.5)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()


def run_regression_sensitivity(df: pd.DataFrame, diseases: Sequence[str], out_path: Path, seed: int) -> None:
    rows: List[Dict[str, object]] = []
    base = df[["moon_phase_sin", "moon_phase_cos", "_doy_sin", "_doy_cos"]].copy()
    valid = base.notna().all(axis=1) & df["_outcome"].notna()
    base = base.loc[valid]

    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(base)
    feat_names = ["moon_phase_sin", "moon_phase_cos", "doy_sin", "doy_cos"]

    for d in diseases:
        y = (df.loc[valid, "_outcome"] == d).astype(int).to_numpy()
        if y.sum() == 0 or y.sum() == len(y):
            continue
        model = LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            random_state=seed,
        )
        model.fit(x_scaled, y)
        for name, coef in zip(feat_names, model.coef_.ravel()):
            rows.append(
                {
                    "disease": d,
                    "term": name,
                    "coef": float(coef),
                    "odds_ratio": float(np.exp(coef)),
                    "p_value": np.nan,
                    "note": "Exploratory sklearn logistic; p-values not estimated in this implementation.",
                }
            )
    pd.DataFrame(rows).to_csv(out_path, index=False)


def write_methods_notes(
    out_path: Path,
    bin_size: int,
    dropped_small: pd.DataFrame,
) -> None:
    lines = [
        "# Lunar Final Methods Notes",
        "",
        "- Input: `outputs/imputed_dataset_enriched.csv`.",
        "- Phase angle basis: `moon_phase_angle_deg` normalized to [0, 360).",
        "- Circular incidence binning: "
        f"{bin_size}° bins with Wilson 95% CI for per-bin risk; moving-average smoothing window = 3 bins.",
        "- Category definition: normalized to `new`, `waxing`, `full`, `waning` "
        "(existing category if mappable, otherwise angle-derived fallback).",
        "- Window sensitivity definitions: "
        "`near_new` and `near_full` using angular windows W in {15°, 30°}.",
        "- RR calculation: risk in level divided by disease baseline prevalence (OvR).",
        "- RR CI: derived using Wilson intervals on level risk and baseline risk, then ratio-bounded.",
        "- Stratification: sex mapped to {male,female,unknown}; neuter mapped to {neutered,intact,unknown}; "
        "combined stratum `sex_neuter` retained for all rows (including unknown).",
        "- Plot/table stratum inclusion rule: strata shown in stratified visuals only if total_n >= 20.",
        f"- Small-strata excluded from stratified visuals: {len(dropped_small)} strata (listed in `sex_neuter_distribution.csv`).",
        "- Caution: imbalance and small cells can widen uncertainty; results are exploratory and non-causal.",
    ]
    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def main() -> None:
    args = parse_args()
    np.random.seed(args.seed)

    input_path = Path(args.input).expanduser().resolve()
    outdir = Path(args.outdir).expanduser().resolve()
    plots_dir = outdir / "plots"
    outdir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)

    df, outcome_col = prepare_dataset(input_path)
    diseases = list(df["_outcome"].value_counts().index)

    sex_neuter_path = outdir / "sex_neuter_distribution.csv"
    dist_df = build_sex_neuter_distribution(df, diseases, sex_neuter_path)
    kept_strata = dist_df.loc[dist_df["reported_stratified"], "sex_neuter"].tolist()
    dropped_small = dist_df.loc[~dist_df["reported_stratified"], ["sex_neuter", "total_n"]]
    dropped_small.to_csv(outdir / "sex_neuter_small_strata.csv", index=False)

    verdict_parts: List[pd.DataFrame] = []
    top_lines: List[Tuple[str, str, float]] = []
    ci_any: Dict[str, bool] = {}

    for disease in diseases:
        disease_slug = slugify(disease)

        circ = circular_curve_table(df, disease, args.bin_size_deg)
        circ.to_csv(outdir / f"moon_circular_curve_{disease_slug}.csv", index=False)
        save_circular_plot(
            circ,
            disease,
            args.bin_size_deg,
            plots_dir / f"moon_circular_curve_{disease_slug}.png",
        )


        rr_cat = compute_category_rr(df, disease, stratum=None)
        rr_win = compute_window_rr(df, disease)
        verdict_parts.extend([rr_cat, rr_win])

        save_rr_barplot(
            rr_cat,
            f"Moon Category RR: {disease}",
            x_col="level",
            out_png=plots_dir / f"moon_rr_category_{disease_slug}.png",
        )

        rr_win_plot = rr_win.copy()
        rr_win_plot["window_label"] = rr_win_plot["metric"] + "_w" + rr_win_plot["window_deg"].astype(int).astype(str)
        save_rr_barplot(
            rr_win_plot,
            f"Moon Window RR: {disease}",
            x_col="window_label",
            out_png=plots_dir / f"moon_rr_windows_{disease_slug}.png",
        )

        rr_cat_sorted = rr_cat.sort_values("rr", ascending=False)
        if not rr_cat_sorted.empty and pd.notna(rr_cat_sorted.iloc[0]["rr"]):
            top_lines.append((disease, str(rr_cat_sorted.iloc[0]["level"]), float(rr_cat_sorted.iloc[0]["rr"])))
        ci_any[disease] = bool(rr_cat["ci_excludes_1"].any() or rr_win["ci_excludes_1"].any())


        strat_rows: List[pd.DataFrame] = []
        for st in kept_strata:
            strat_rows.append(compute_category_rr(df, disease, stratum=st))
        if strat_rows:
            rr_strat = pd.concat(strat_rows, ignore_index=True)
            verdict_parts.append(rr_strat)
            save_stratified_heatmap(
                rr_strat,
                disease,
                kept_strata,
                plots_dir / f"moon_rr_stratified_heatmap_{disease_slug}.png",
            )

    verdict = pd.concat(verdict_parts, ignore_index=True).sort_values(
        ["disease", "analysis_level", "stratum", "metric", "level", "window_deg"],
        kind="mergesort",
    )
    verdict.to_csv(outdir / "moon_verdict_table.csv", index=False)


    run_regression_sensitivity(df, diseases, outdir / "moon_regression_sensitivity.csv", seed=args.seed)

    write_methods_notes(outdir / "moon_methods_notes.md", args.bin_size_deg, dropped_small)


    print("\nTop RR category per disease:")
    for d, lvl, rr in top_lines:
        print(f"- {d}: {lvl} (RR={rr:.3f})")
    print("\nAny RR CI excludes 1 (category/window):")
    for d in diseases:
        print(f"- {d}: {'yes' if ci_any.get(d, False) else 'no'}")
    print("\nUnknown stratum counts:")
    unknown_rows = dist_df[dist_df["sex_neuter"].str.contains("unknown", na=False)]
    if unknown_rows.empty:
        print("- none")
    else:
        for _, r in unknown_rows.sort_values("total_n", ascending=False).iterrows():
            print(f"- {r['sex_neuter']}: n={int(r['total_n'])}")

    print(f"\nOutputs written to: {outdir}")
    print(f"Outcome column used: {outcome_col}")


if __name__ == "__main__":
    main()
